In [1]:
!pip install pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

spark = SparkSession.builder \
    .appName("Week6_Assignment_Nomaan") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(f'Spark Version: {spark.version}')
print('SparkSession ready ✅')

Spark Version: 4.0.3
SparkSession ready ✅


## Dataset Generation:
I have taken the help of AI to synthesize an appropriate dataset to perform all the questions in the assignment effectively.

In [2]:
import csv
import random

random.seed(42)

categories = ["Electronics", "Grocery", "Clothing", "Furniture", "Toys"]
regions = ["North", "South", "East", "West"]
priorities = ["High", "Medium", "Low"]
statuses = ["Completed", "Pending", "Cancelled"]

rows = []
for i in range(1, 501):
    product_id = f"P{i:04d}"
    category = random.choice(categories)
    base_price = round(random.uniform(50, 5000), 2)
    user_id = None if random.random() < 0.05 else f"U{random.randint(1000,1999)}"
    region = random.choice(regions)
    priority = random.choice(priorities)
    status = random.choice(statuses)
    amount = round(base_price * random.uniform(1, 3), 2)
    price = str(round(base_price * random.uniform(0.9, 1.1), 2))
    old_name = f"item_{i}"
    rows.append([product_id, old_name, category, price, base_price, user_id, region, priority, status, amount])

header = ["product_id", "old_name", "category", "price", "base_price", "user_id", "region", "priority", "status", "amount"]

with open("source.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print("CSV generated with", len(rows), "rows")

CSV generated with 500 rows


##Q3: Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

In [3]:
df = spark.read.csv("source.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)

+----------+--------+-----------+-------+----------+-------+------+--------+---------+-------+
|product_id|old_name|   category|  price|base_price|user_id|region|priority|   status| amount|
+----------+--------+-----------+-------+----------+-------+------+--------+---------+-------+
|     P0001|  item_1|Electronics| 187.43|     173.8|  U1228| South|     Low|Completed| 409.02|
|     P0002|  item_2|Electronics|2691.42|   2972.94|   NULL| North|    High|Completed|5977.72|
|     P0003|  item_3|    Grocery|3859.78|    3594.3|  U1429| South|  Medium|Cancelled| 5594.1|
|     P0004|  item_4|    Groce

### Insight:
When Spark reads the CSV file, it splita the work across multiple executors to scan the file.But before it can load the data it needs to know the type of eac column of the table.
Here I have written 'inferSchema' which is like asking Spark to guess the type of column by itslef by making it read a few columns which makes the whole process a bit slower.
This can be optimized by explicitly defining the schema because on huge datasets the repeated guessing would create a considreable overhead.

##Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

In [4]:
result_q5 = df.select("product_id", "price").filter(col("category") == "Electronics")
result_q5.show(5)
print(result_q5.count())

+----------+-------+
|product_id|  price|
+----------+-------+
|     P0001| 187.43|
|     P0002|2691.42|
|     P0019|3709.53|
|     P0023|3988.61|
|     P0026|1216.76|
+----------+-------+
only showing top 5 rows
98


###Insight:
The filter and select are both narrow transformations , Spark can apply them independently on each partition without needing to shuffle data between executors.
Spark's optimizer can reorder the filter to run before the column selection, so it removes non-Electronics rows early instead of processing all columns first.

##Q6: Write the code to "revise" a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

In [5]:
df_revised = df.withColumnRenamed("old_name", "new_name")
df_revised = df_revised.withColumn("price", col("price").cast(DoubleType()))

df_revised.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)



###Insight:
withColumnRenamed and .cast() are row-level operations so they don't require moving data between executors, so no shuffle happens here. Each partition handles its own rows independently.

##Q8: Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

In [6]:
from pyspark.sql.functions import col

df_orders = df_revised
result_q8 = df_orders.filter((col("status") == "Completed") & (col("amount") > 1000))
result_q8.show(5)
print(result_q8.count())

+----------+--------+-----------+-------+----------+-------+------+--------+---------+--------+
|product_id|new_name|   category|  price|base_price|user_id|region|priority|   status|  amount|
+----------+--------+-----------+-------+----------+-------+------+--------+---------+--------+
|     P0002|  item_2|Electronics|2691.42|   2972.94|   NULL| North|    High|Completed| 5977.72|
|     P0004|  item_4|    Grocery|3223.02|   3505.79|  U1159| South|  Medium|Completed| 4156.08|
|     P0005|  item_5|   Clothing|2964.63|   3038.44|  U1747|  West|     Low|Completed| 8951.95|
|     P0006|  item_6|       Toys|1364.87|   1501.23|  U1906|  East|     Low|Completed| 3616.68|
|     P0018| item_18|   Clothing|4792.96|   4399.67|  U1314| South|    High|Completed|12125.57|
+----------+--------+-----------+-------+----------+-------+------+--------+---------+--------+
only showing top 5 rows
143


###Insights:
This is a row-level transformation : filtering with & (AND) evaluates row-by-row within each partition independently, so no shuffle is needed between executors.

##Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [7]:
result_q10 = df_revised.withColumn("final_price",col("base_price") * 1.18)
result_q10.select("product_id", "base_price", "final_price").show(5)

+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|     P0001|     173.8|    205.084|
|     P0002|   2972.94|  3508.0692|
|     P0003|    3594.3|   4241.274|
|     P0004|   3505.79|  4136.8322|
|     P0005|   3038.44|  3585.3592|
+----------+----------+-----------+
only showing top 5 rows


###Insights:
withColumn here is a row-level transformation so each executor computes final_price for its own partition independently, with no shuffle or cross-executor communication needed.

##Q12: Write the Spark command to load a Parquet file from "path/to/input", filter out any rows where user_id is null, and save the result as a CSV at "path/to/output".

In [8]:
result_q10.write.mode("overwrite").parquet("source_parquet")

df_parquet = spark.read.parquet("source_parquet")
before_count = df_parquet.count()

result_q12 = df_parquet.filter(col("user_id").isNotNull())
after_count = result_q12.count()

result_q12.write.mode("overwrite").option("header", True).csv("cleaned_output")

print("Before:", before_count)
print("After:", after_count)
print("Nulls removed:", before_count - after_count)

#Just to verify that the csv file has no nulls remaining
df_check = spark.read.csv("cleaned_output", header=True, inferSchema=True)
null_count = df_check.filter(col("user_id").isNull()).count()
print("Nulls remaining:", null_count)

Before: 500
After: 476
Nulls removed: 24
Nulls remaining: 0


###Insgihts:
This check re-reads the CSV output we just wrote (cleaned_output) as a fresh DataFrame, then applies a filter and a count() action. This is a full read > filter > count cycle,which means Spark has to scan the written files again from disk — it's not reusing anything cached in memory from the earlier steps.

##Q14: Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.


In [9]:
result_q14 = df_revised.filter((col("region") == "North") | (col("priority") == "High"))
result_q14.show(5)
print(result_q14.count())

+----------+--------+-----------+-------+----------+-------+------+--------+---------+-------+
|product_id|new_name|   category|  price|base_price|user_id|region|priority|   status| amount|
+----------+--------+-----------+-------+----------+-------+------+--------+---------+-------+
|     P0002|  item_2|Electronics|2691.42|   2972.94|   NULL| North|    High|Completed|5977.72|
|     P0007|  item_7|    Grocery|3981.89|   3876.69|  U1875| South|    High|  Pending|6031.92|
|     P0010| item_10|   Clothing|3978.07|   4222.12|  U1234| North|  Medium|  Pending|6482.99|
|     P0013| item_13|    Grocery|2665.01|   2572.16|  U1048| North|    High|Cancelled|3395.16|
|     P0015| item_15|       Toys|3392.61|   3766.85|  U1348| North|  Medium|  Pending|4958.36|
+----------+--------+-----------+-------+----------+-------+------+--------+---------+-------+
only showing top 5 rows
240


###Insights:
This is a row-level transformation, same kind as Q8 — the OR condition (|) is evaluated row-wise within each partition independently, so no shuffle is needed across executors.